In [1688]:
import pandas as pd
import regex as re
import io

In [1689]:
def read_vcf(path):
    with open(path, 'r') as f:
        lines = [l for l in f if not l.startswith('##')]
    return pd.read_csv(
        io.StringIO(''.join(lines)),
        dtype={'#CHROM': str, 'POS': int, 'ID': str, 'REF': str, 'ALT': str,
               'QUAL': str, 'FILTER': str, 'INFO': str},
        sep='\t'
    ).rename(columns={'#CHROM': 'CHROM'})

def extract_af(info):
    match = re.search(r'AF=([\d\.]+)', info)
    return match.group(1) if match else None

In [1690]:
df = read_vcf('/Users/fionachow/Documents/NYU/CDS/Spring 2024/Research Fair/Hochwagen/Individual Data/ERR3240115/ERR3240115_rDNA.vcf') #vcf file from original variant calling of 1 individual
df['AF'] = df['INFO'].apply(extract_af)

df.head(60)

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,AF
0,Human,31,.,T,C,21247,PASS,"DP=1510;AF=1.000000;SB=0;DP4=0,0,1509,1",1.000000
1,Human,32,.,C,T,22076,PASS,"DP=1554;AF=1.000000;SB=0;DP4=0,0,1553,1",1.000000
2,Human,73,.,C,A,46496,PASS,"DP=2625;AF=0.996952;SB=0;DP4=7,0,2607,10",0.996952
3,Human,74,.,A,C,48045,PASS,"DP=2657;AF=0.999624;SB=0;DP4=1,0,2646,10",0.999624
4,Human,80,.,A,AC,49314,PASS,"DP=2829;AF=0.977024;SB=0;DP4=64,0,2747,17;INDE...",0.977024
5,Human,103,.,GC,G,49314,PASS,"DP=3376;AF=0.987559;SB=31;DP4=38,4,3304,30;IND...",0.987559
6,Human,121,.,C,G,49314,PASS,"DP=3853;AF=0.998962;SB=0;DP4=0,0,3779,70",0.998962
7,Human,209,.,C,G,2469,PASS,"DP=4542;AF=0.109203;SB=2;DP4=3584,459,436,60",0.109203
8,Human,237,.,G,GC,49314,PASS,"DP=4988;AF=0.966921;SB=14;DP4=164,21,3988,835;...",0.966921
9,Human,270,.,C,CG,49314,PASS,"DP=5712;AF=0.945553;SB=6;DP4=242,88,4122,1279;...",0.945553


In [1691]:
type(df['AF'].iloc[0])

str

In [1692]:
#get sequnce of individual at each position
og_ref = pd.read_csv('/Users/fionachow/Documents/NYU/CDS/Summer 2024/Human rDNA Research/Project/Human-rDNA/outputs/src_outputs/og_ref.csv')

og_ref.shape

(13314, 3)

In [1693]:
new_ref = pd.read_csv('/Users/fionachow/Documents/NYU/CDS/Summer 2024/Human rDNA Research/Project/Human-rDNA/outputs/src_outputs/1000_genome_new_ref/1000_genome_new_ref_v2.csv')

new_ref.shape

(13314, 3)

In [1694]:
df = df[['POS', 'REF', 'ALT', 'AF']]  

print(og_ref.columns)
print(df.columns)

merged_df = pd.merge(og_ref, df, on='POS', how='left', suffixes=('_og', '_vcf'))

merged_df.rename(columns={'REF_og':'ALT_og', 'ALT':'ALT_vcf'}, inplace=True)

merged_df.head(33)

Index(['POS', 'Type', 'REF'], dtype='object')
Index(['POS', 'REF', 'ALT', 'AF'], dtype='object')


,POS,Type,ALT_og,REF_vcf,ALT_vcf,AF
0,1,sequence,G,NaN,NaN,NaN
1,2,sequence,C,NaN,NaN,NaN
2,3,sequence,T,NaN,NaN,NaN
3,4,sequence,G,NaN,NaN,NaN
4,5,sequence,A,NaN,NaN,NaN
5,6,sequence,C,NaN,NaN,NaN
6,7,sequence,A,NaN,NaN,NaN
7,8,sequence,C,NaN,NaN,NaN
8,9,sequence,G,NaN,NaN,NaN
9,10,sequence,C,NaN,NaN,NaN


In [1695]:
merged_df.shape

(13346, 6)

In [1696]:
merged_df[(merged_df['POS'] - 1)!= merged_df.index]

,POS,Type,ALT_og,REF_vcf,ALT_vcf,AF
386,386,sequence,T,T,TGGCC,0.736424
387,387,sequence,G,NaN,NaN,NaN
388,388,sequence,G,NaN,NaN,NaN
389,389,sequence,C,NaN,NaN,NaN
390,390,sequence,C,NaN,NaN,NaN
...,...,...,...,...,...,...
13341,13310,sequence,T,NaN,NaN,NaN
13342,13311,sequence,C,NaN,NaN,NaN
13343,13312,sequence,G,NaN,NaN,NaN
13344,13313,sequence,C,NaN,NaN,NaN


In [1697]:
merged_df.head(389)

,POS,Type,ALT_og,REF_vcf,ALT_vcf,AF
0,1,sequence,G,NaN,NaN,NaN
1,2,sequence,C,NaN,NaN,NaN
2,3,sequence,T,NaN,NaN,NaN
3,4,sequence,G,NaN,NaN,NaN
4,5,sequence,A,NaN,NaN,NaN
...,...,...,...,...,...,...
384,385,sequence,C,NaN,NaN,NaN
385,386,sequence,T,T,TGGCAA,0.100669
386,386,sequence,T,T,TGGCC,0.736424
387,387,sequence,G,NaN,NaN,NaN


In [1698]:
df[df['POS'] == 386]

,POS,REF,ALT,AF
13,386,T,TGGCAA,0.100669
14,386,T,TGGCC,0.736424


In [1699]:
# Combine the ALT_og column and ALT_vcf into ALT_combined, and set AF to 0 where variant did not exist
merged_df['ALT_combined'] = merged_df.apply(lambda x: x['ALT_vcf'] if pd.notna(x['ALT_vcf']) else x['ALT_og'], axis=1)
merged_df['AF'] = merged_df['AF'].fillna(0) 

merged_df.head(32)

,POS,Type,ALT_og,REF_vcf,ALT_vcf,AF,ALT_combined
0,1,sequence,G,NaN,NaN,0,G
1,2,sequence,C,NaN,NaN,0,C
2,3,sequence,T,NaN,NaN,0,T
3,4,sequence,G,NaN,NaN,0,G
4,5,sequence,A,NaN,NaN,0,A
5,6,sequence,C,NaN,NaN,0,C
6,7,sequence,A,NaN,NaN,0,A
7,8,sequence,C,NaN,NaN,0,C
8,9,sequence,G,NaN,NaN,0,G
9,10,sequence,C,NaN,NaN,0,C


In [1700]:
merged_df.head(80) 

,POS,Type,ALT_og,REF_vcf,ALT_vcf,AF,ALT_combined
0,1,sequence,G,NaN,NaN,0,G
1,2,sequence,C,NaN,NaN,0,C
2,3,sequence,T,NaN,NaN,0,T
3,4,sequence,G,NaN,NaN,0,G
4,5,sequence,A,NaN,NaN,0,A
...,...,...,...,...,...,...,...
75,76,sequence,G,NaN,NaN,0,G
76,77,sequence,G,NaN,NaN,0,G
77,78,sequence,T,NaN,NaN,0,T
78,79,sequence,G,NaN,NaN,0,G


In [1701]:
merged_df.head(106) 



,POS,Type,ALT_og,REF_vcf,ALT_vcf,AF,ALT_combined
0,1,sequence,G,NaN,NaN,0,G
1,2,sequence,C,NaN,NaN,0,C
2,3,sequence,T,NaN,NaN,0,T
3,4,sequence,G,NaN,NaN,0,G
4,5,sequence,A,NaN,NaN,0,A
...,...,...,...,...,...,...,...
101,102,sequence,T,NaN,NaN,0,T
102,103,sequence,G,GC,G,0.987559,G
103,104,sequence,C,NaN,NaN,0,C
104,105,sequence,C,NaN,NaN,0,C


In [1702]:
###to identify deletions from this table 
###-> Alt_og == Alt_vcf (get position)
###-> save position if any of subsequent positions are NaN in Alt_vcf but not Nan in Alt_og 
   ### --> add the base and position to a deletion_dictionary = {104: 'C'}

In [1703]:
merged_df.drop(columns=['ALT_vcf','Type'], inplace=True)

merged_df.rename(columns={'ALT_og': 'og_ref', 'AF':'AF_old', 'ALT_combined':'ALT_old'}, inplace=True)

merged_df.head(73)

,POS,og_ref,REF_vcf,AF_old,ALT_old
0,1,G,NaN,0,G
1,2,C,NaN,0,C
2,3,T,NaN,0,T
3,4,G,NaN,0,G
4,5,A,NaN,0,A
...,...,...,...,...,...
68,69,G,NaN,0,G
69,70,C,NaN,0,C
70,71,C,NaN,0,C
71,72,T,NaN,0,T


In [1704]:
merged_df = pd.merge(merged_df, new_ref, on='POS', how='left')
merged_df.drop(columns=['Unnamed: 0'], inplace=True)
merged_df.rename(columns={'1000_genome_new_ref': 'new_ref'}, inplace=True)

merged_df

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref
0,1,G,NaN,0,G,G
1,2,C,NaN,0,C,C
2,3,T,NaN,0,T,T
3,4,G,NaN,0,G,G
4,5,A,NaN,0,A,A
...,...,...,...,...,...,...
13341,13310,T,NaN,0,T,T
13342,13311,C,NaN,0,C,C
13343,13312,G,NaN,0,G,G
13344,13313,C,NaN,0,C,C


In [1705]:
merged_df.head(1676)

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref
0,1,G,NaN,0,G,G
1,2,C,NaN,0,C,C
2,3,T,NaN,0,T,T
3,4,G,NaN,0,G,G
4,5,A,NaN,0,A,A
...,...,...,...,...,...,...
1671,1670,C,NaN,0,C,C
1672,1671,G,GC,0.031593,G,G
1673,1671,G,GCTCCC,0.596132,G,G
1674,1672,C,NaN,0,C,0


In [1706]:
merged_df.head(104)

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref
0,1,G,NaN,0,G,G
1,2,C,NaN,0,C,C
2,3,T,NaN,0,T,T
3,4,G,NaN,0,G,G
4,5,A,NaN,0,A,A
...,...,...,...,...,...,...
99,100,C,NaN,0,C,C
100,101,C,NaN,0,C,C
101,102,T,NaN,0,T,T
102,103,G,GC,0.987559,G,G


In [1707]:
merged_df.head(636)

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref
0,1,G,NaN,0,G,G
1,2,C,NaN,0,C,C
2,3,T,NaN,0,T,T
3,4,G,NaN,0,G,G
4,5,A,NaN,0,A,A
...,...,...,...,...,...,...
631,631,G,NaN,0,G,G
632,632,G,NaN,0,G,G
633,633,G,GT,0.052245,G,G
634,634,T,NaN,0,T,T


In [1708]:
merged_df.head(1676)

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref
0,1,G,NaN,0,G,G
1,2,C,NaN,0,C,C
2,3,T,NaN,0,T,T
3,4,G,NaN,0,G,G
4,5,A,NaN,0,A,A
...,...,...,...,...,...,...
1671,1670,C,NaN,0,C,C
1672,1671,G,GC,0.031593,G,G
1673,1671,G,GCTCCC,0.596132,G,G
1674,1672,C,NaN,0,C,0


In [1709]:
merged_df[(merged_df['new_ref'].str.len()) > (merged_df['og_ref'].str.len())]

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref
79,80,A,A,0.977024,AC,AC
236,237,G,G,0.966921,GC,GC
269,270,C,C,0.945553,CG,CG
274,275,C,C,0.937946,CG,CG
283,284,A,A,0.960138,AC,AC
362,363,G,G,0.905516,GT,GT
385,386,T,T,0.100669,TGGCAA,TGGCC
386,386,T,T,0.736424,TGGCC,TGGCC
419,419,C,C,0.925278,CGT,CGT
517,517,G,G,0.940377,GC,GC


In [1710]:
filtered_df = merged_df[(merged_df['og_ref'] != merged_df['new_ref']) & (merged_df['AF_old'] != 0)]

filtered_df.shape #number of ALT rows being changed

(87, 6)

In [1711]:
filtered_df

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref
30,31,T,T,1.000000,C,C
31,32,C,C,1.000000,T,T
72,73,C,C,0.996952,A,A
73,74,A,A,0.999624,C,C
79,80,A,A,0.977024,AC,AC
...,...,...,...,...,...,...
13181,13150,A,A,0.961884,AC,AC
13250,13219,C,C,0.944088,CG,CG
13309,13278,G,G,0.909971,GGC,GGC
13311,13280,G,G,1.000000,C,C


In [1712]:
merged_df['REF_vcf'].iloc[0]

nan

In [1713]:
#incorporating deletions iff it exists
merged_df['new_ref'] = merged_df.apply(lambda x: x['REF_vcf'] if pd.notna(x['REF_vcf']) and (len(x['REF_vcf']) > 1)  else x['new_ref'], axis=1)

merged_df.head(1676)

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref
0,1,G,NaN,0,G,G
1,2,C,NaN,0,C,C
2,3,T,NaN,0,T,T
3,4,G,NaN,0,G,G
4,5,A,NaN,0,A,A
...,...,...,...,...,...,...
1671,1670,C,NaN,0,C,C
1672,1671,G,GC,0.031593,G,GC
1673,1671,G,GCTCCC,0.596132,G,GCTCCC
1674,1672,C,NaN,0,C,0


In [1714]:
merged_df.head(635)

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref
0,1,G,NaN,0,G,G
1,2,C,NaN,0,C,C
2,3,T,NaN,0,T,T
3,4,G,NaN,0,G,G
4,5,A,NaN,0,A,A
...,...,...,...,...,...,...
630,630,C,NaN,0,C,C
631,631,G,NaN,0,G,G
632,632,G,NaN,0,G,G
633,633,G,GT,0.052245,G,GT


In [1715]:
merged_df = merged_df[merged_df['new_ref']!='0']

merged_df.head(104)

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref
0,1,G,NaN,0,G,G
1,2,C,NaN,0,C,C
2,3,T,NaN,0,T,T
3,4,G,NaN,0,G,G
4,5,A,NaN,0,A,A
...,...,...,...,...,...,...
99,100,C,NaN,0,C,C
100,101,C,NaN,0,C,C
101,102,T,NaN,0,T,T
102,103,G,GC,0.987559,G,GC


In [1716]:
#using ALT_old unless reference changed and AF_old != 0
merged_df['ALT_new'] = merged_df.apply(lambda x: x['og_ref'] if (x['AF_old'] != 0) and (x['og_ref'] != x['new_ref']) else x['ALT_old'], axis=1)

merged_df

/var/folders/hb/l21bg8lx3tgc5ky3s56cwdth0000gn/T/ipykernel_69148/1180185392.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_df['ALT_new'] = merged_df.apply(lambda x: x['og_ref'] if (x['AF_old'] != 0) and (x['og_ref'] != x['new_ref']) else x['ALT_old'], axis=1)


,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new
0,1,G,NaN,0,G,G,G
1,2,C,NaN,0,C,C,C
2,3,T,NaN,0,T,T,T
3,4,G,NaN,0,G,G,G
4,5,A,NaN,0,A,A,A
...,...,...,...,...,...,...,...
13341,13310,T,NaN,0,T,T,T
13342,13311,C,NaN,0,C,C,C
13343,13312,G,NaN,0,G,G,G
13344,13313,C,NaN,0,C,C,C


In [1717]:
merged_df.head(32)

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new
0,1,G,NaN,0,G,G,G
1,2,C,NaN,0,C,C,C
2,3,T,NaN,0,T,T,T
3,4,G,NaN,0,G,G,G
4,5,A,NaN,0,A,A,A
5,6,C,NaN,0,C,C,C
6,7,A,NaN,0,A,A,A
7,8,C,NaN,0,C,C,C
8,9,G,NaN,0,G,G,G
9,10,C,NaN,0,C,C,C


In [1718]:
merged_df.head(104)

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new
0,1,G,NaN,0,G,G,G
1,2,C,NaN,0,C,C,C
2,3,T,NaN,0,T,T,T
3,4,G,NaN,0,G,G,G
4,5,A,NaN,0,A,A,A
...,...,...,...,...,...,...,...
99,100,C,NaN,0,C,C,C
100,101,C,NaN,0,C,C,C
101,102,T,NaN,0,T,T,T
102,103,G,GC,0.987559,G,GC,G


In [1719]:
merged_df.head(636)

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new
0,1,G,NaN,0,G,G,G
1,2,C,NaN,0,C,C,C
2,3,T,NaN,0,T,T,T
3,4,G,NaN,0,G,G,G
4,5,A,NaN,0,A,A,A
...,...,...,...,...,...,...,...
633,633,G,GT,0.052245,G,GT,G
634,634,T,NaN,0,T,T,T
635,635,G,NaN,0,G,G,G
636,636,C,NaN,0,C,C,C


In [1720]:
filtered_df2 = merged_df[(merged_df['og_ref'] != merged_df['new_ref'])]

filtered_df2.shape #number of AF rows being changed

(148, 7)

In [1721]:
#using AF_old unless reference changed 
merged_df['AF_new'] = merged_df.apply(lambda x: 1 - float(x['AF_old']) if (x['og_ref'] != x['new_ref']) and (x['REF_vcf'] != x['new_ref']) else float(x['AF_old']), axis=1) #note: 'AF_new' is all in float, 'AF_old' was in string and 0 was integer

merged_df

/var/folders/hb/l21bg8lx3tgc5ky3s56cwdth0000gn/T/ipykernel_69148/932589795.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_df['AF_new'] = merged_df.apply(lambda x: 1 - float(x['AF_old']) if (x['og_ref'] != x['new_ref']) and (x['REF_vcf'] != x['new_ref']) else float(x['AF_old']), axis=1) #note: 'AF_new' is all in float, 'AF_old' was in string and 0 was integer


,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new,AF_new
0,1,G,NaN,0,G,G,G,0.0
1,2,C,NaN,0,C,C,C,0.0
2,3,T,NaN,0,T,T,T,0.0
3,4,G,NaN,0,G,G,G,0.0
4,5,A,NaN,0,A,A,A,0.0
...,...,...,...,...,...,...,...,...
13341,13310,T,NaN,0,T,T,T,0.0
13342,13311,C,NaN,0,C,C,C,0.0
13343,13312,G,NaN,0,G,G,G,0.0
13344,13313,C,NaN,0,C,C,C,0.0


In [1722]:
merged_df.iloc[30, :]

POS              31
og_ref            T
REF_vcf           T
AF_old     1.000000
ALT_old           C
new_ref           C
ALT_new           T
AF_new          0.0
Name: 30, dtype: object

In [1723]:
merged_df[merged_df['AF_new']!=0].head(50) #new vcf output

,POS,og_ref,REF_vcf,AF_old,ALT_old,new_ref,ALT_new,AF_new
72,73,C,C,0.996952,A,A,C,0.003048
73,74,A,A,0.999624,C,C,A,0.000376
79,80,A,A,0.977024,AC,AC,A,0.022976
102,103,G,GC,0.987559,G,GC,G,0.987559
120,121,C,C,0.998962,G,G,C,0.001038
208,209,C,C,0.109203,G,C,G,0.109203
236,237,G,G,0.966921,GC,GC,G,0.033079
269,270,C,C,0.945553,CG,CG,C,0.054447
274,275,C,C,0.937946,CG,CG,C,0.062054
283,284,A,A,0.960138,AC,AC,A,0.039862
